Bob has started his own mobile company. He wants to give tough fight to big companies 
like Apple, Samsung etc. He does not know how to estimate price of mobiles his company 
creates. In this competitive mobile phone market, you cannot simply assume things. To 
solve this problem, he collects sales data of mobile phones of various companies. Bob 
wants to find out some relation between features of a mobile phone (eg:- RAM, Internal 
Memory etc) and its selling price. But he is not so good at Machine Learning. Create an 
artificial neural network-based classifier to help Bob. Data and class labels are available 
in mobile_price_classification.csv file. Use price_range column as label and remaining 
columns as data. 
Also do the hyperparmeter tuning. 

Import Required Libraries

In [3]:
!pip install tensorflow

   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.8/332.0 MB 6.3 MB/s eta 0:00:53
   ---------------------------------------- 3.1/332.0 MB 9.4 MB/s eta 0:00:36
    --------------------------------------- 5.5/332.0 MB 10.0 MB/s eta 0:00:33
   - -------------------------------------- 8.7/332.0 MB 11.4 MB/s eta 0:00:29
   - -------------------------------------- 12.1/332.0 MB 12.5 MB/s eta 0:00:26
   - -------------------------------------- 15.7/332.0 MB 13.2 MB/s eta 0:00:24
   -- ------------------------------------- 19.9/332.0 MB 14.3 MB/s eta 0:00:22
   --- ------------------------------------ 24.9/332.0 MB 15.6 MB/s eta 0:00:20
   --- ------------------------------------ 28.3/332.0 MB 15.6 MB/s eta 0:00:20
   ---- ----------------------------------- 34.3/332.0 MB 17.0 MB/s eta 0:00:18
   ---- ----------------------------------- 41.4/332.0 MB 18.4 MB/s eta 0:00:16
   ----- ---------------------------------- 49.5/332.0 

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

Load the Dataset

In [2]:
data = pd.read_csv("mobile_price_classification.csv")
data.head()

,battery_power,bluetooth,clock_speed,dual_sim,front_cam,4G,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [18]:
data.shape

(2000, 21)

In [19]:
data.info() #data set information

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   battery_power   2000 non-null   int64  
 1   bluetooth       2000 non-null   int64  
 2   clock_speed     2000 non-null   float64
 3   dual_sim        2000 non-null   int64  
 4   front_cam       2000 non-null   int64  
 5   4G              2000 non-null   int64  
 6   int_memory      2000 non-null   int64  
 7   m_dep           2000 non-null   float64
 8   mobile_wt       2000 non-null   int64  
 9   n_cores         2000 non-null   int64  
 10  primary_camera  2000 non-null   int64  
 11  px_height       2000 non-null   int64  
 12  px_width        2000 non-null   int64  
 13  ram             2000 non-null   int64  
 14  sc_h            2000 non-null   int64  
 15  sc_w            2000 non-null   int64  
 16  talk_time       2000 non-null   int64  
 17  three_g         2000 non-null   i

In [20]:
data.describe()  # statistical summary

,battery_power,bluetooth,clock_speed,dual_sim,front_cam,4G,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
count,2000.000000,2000.0000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,...,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,1238.518500,0.4950,1.522250,0.509500,4.309500,0.521500,32.046500,0.501750,140.249000,4.520500,...,645.108000,1251.515500,2124.213000,12.306500,5.767000,11.011000,0.761500,0.503000,0.507000,1.500000
std,439.418206,0.5001,0.816004,0.500035,4.341444,0.499662,18.145715,0.288416,35.399655,2.287837,...,443.780811,432.199447,1084.732044,4.213245,4.356398,5.463955,0.426273,0.500116,0.500076,1.118314
min,501.000000,0.0000,0.500000,0.000000,0.000000,0.000000,2.000000,0.100000,80.000000,1.000000,...,0.000000,500.000000,256.000000,5.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000
25%,851.750000,0.0000,0.700000,0.000000,1.000000,0.000000,16.000000,0.200000,109.000000,3.000000,...,282.750000,874.750000,1207.500000,9.000000,2.000000,6.000000,1.000000,0.000000,0.000000,0.750000
50%,1226.000000,0.0000,1.500000,1.000000,3.000000,1.000000,32.000000,0.500000,141.000000,4.000000,...,564.000000,1247.000000,2146.500000,12.000000,5.000000,11.000000,1.000000,1.000000,1.000000,1.500000
75%,1615.250000,1.0000,2.200000,1.000000,7.000000,1.000000,48.000000,0.800000,170.000000,7.000000,...,947.250000,1633.000000,3064.500000,16.000000,9.000000,16.000000,1.000000,1.000000,1.000000,2.250000
max,1998.000000,1.0000,3.000000,1.000000,19.000000,1.000000,64.000000,1.000000,200.000000,8.000000,...,1960.000000,1998.000000,3998.000000,19.000000,18.000000,20.000000,1.000000,1.000000,1.000000,3.000000


In [21]:
data.isnull().sum()   #check missing values

battery_power     0
bluetooth         0
clock_speed       0
dual_sim          0
front_cam         0
4G                0
int_memory        0
m_dep             0
mobile_wt         0
n_cores           0
primary_camera    0
px_height         0
px_width          0
ram               0
sc_h              0
sc_w              0
talk_time         0
three_g           0
touch_screen      0
wifi              0
price_range       0
dtype: int64

In [22]:
data['price_range'].value_counts()  #check distribution of target

price_range
1    500
2    500
3    500
0    500
Name: count, dtype: int64

In [23]:
X = data.drop('price_range', axis=1)
y = data['price_range']
#feature and target separation

Data Preprocessing

Separate features and target variable.

In [26]:
X = data.drop("price_range", axis=1)
Y = data["price_range"]

print(X.shape)
print(Y.shape)

(2000, 20)
(2000,)


Split the Dataset

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42)

Feature Scaling

In [8]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Build the Artificial Neural Network Model

In this step, an Artificial Neural Network (ANN) model is created to classify the mobile price range based on its features.
The model consists of an input layer, hidden layers with ReLU activation, and an output layer with Softmax activation.
The Softmax function is used because the target variable contains multiple classes representing different price ranges.

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

model = Sequential()

model.add(Input(shape=(X_train.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(4, activation='softmax'))

Compile the Model

In [12]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

Train the Model

In [13]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.3203 - loss: 1.3721 - val_accuracy: 0.4187 - val_loss: 1.2922
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5250 - loss: 1.1753 - val_accuracy: 0.5469 - val_loss: 1.1177
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6383 - loss: 0.9605 - val_accuracy: 0.6500 - val_loss: 0.9104
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7453 - loss: 0.7601 - val_accuracy: 0.7250 - val_loss: 0.7372
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8062 - loss: 0.6083 - val_accuracy: 0.8031 - val_loss: 0.6083
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8648 - loss: 0.4971 - val_accuracy: 0.8500 - val_loss: 0.5101
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9055 - loss: 0.4073 - val_accuracy: 0.8656 - val_loss: 0.4362
Epoch 8/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9234 - loss: 0.3406 - val_accuracy: 0.8969 - val_loss:

Model Evaluation

In [14]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9300 - loss: 0.1837  
Test Accuracy: 0.9300000071525574


Prediction

In [15]:
y_pred = model.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
Accuracy: 0.93
              precision    recall  f1-score   support

           0       0.97      0.95      0.96       105
           1       0.86      0.95      0.90        91
           2       0.93      0.85      0.89        92
           3       0.96      0.96      0.96       112

    accuracy                           0.93       400
   macro avg       0.93      0.93      0.93       400
weighted avg       0.93      0.93      0.93       400



Hyperparameter Tuning

Different parameters can improve the model performance such as:

Number of hidden layers

Number of neurons

Epochs

Batch size

Example model with different parameters:

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

model2 = Sequential()

model2.add(Input(shape=(X_train.shape[1],)))
model2.add(Dense(256, activation='relu'))
model2.add(Dense(128, activation='relu'))
model2.add(Dense(64, activation='relu'))
model2.add(Dense(4, activation='softmax'))

model2.compile(
    optimizer='rmsprop',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model2.fit(
    X_train,
    y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5813 - loss: 1.0145 - val_accuracy: 0.6906 - val_loss: 0.6716
Epoch 2/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8648 - loss: 0.4508 - val_accuracy: 0.8813 - val_loss: 0.3475
Epoch 3/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9156 - loss: 0.2546 - val_accuracy: 0.9062 - val_loss: 0.2559
Epoch 4/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9430 - loss: 0.1770 - val_accuracy: 0.9187 - val_loss: 0.2141
Epoch 5/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9555 - loss: 0.1337 - val_accuracy: 0.8875 - val_loss: 0.2637
Epoch 6/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9633 - loss: 0.1127 - val_accuracy: 0.9125 - val_loss: 0.2036
Epoch 7/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9797 - loss: 0.0811 - val_accuracy: 0.9250 - val_loss: 0.2114
Epoch 8/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9703 - loss: 0.0769 - val_accuracy: 0.9094 - val_loss:

Conclusion

In this case study, an Artificial Neural Network was developed to predict the price range of mobile phones based on their features.
The model was trained and evaluated using machine learning techniques. Hyperparameter tuning helped improve the performance of the model. The final model successfully classified mobile phones into different price categories.